# ClipSAT Question Bank Auditor, SymPy Verifier & Past-Paper Generator
### Powered by Your Built-In Google Colab Ultra Subscription (`google.colab.ai`)

This notebook audits, repairs, and expands any ClipSAT question bank (`bank-data/<track>.json`) using:
1. **Built-in `google.colab.ai` (`google/gemini-3.1-pro-preview` / `google/gemini-3.8-flash`)** — uses your Google Ultra subscription directly with **no paid API key required**.
2. **SymPy Symbolic Math Engine** — executes Python verification code on every single question before it is accepted, guaranteeing that the correct choice is mathematically exact and unique.
3. **Strict Past-Paper Curriculum Taxonomy (`curriculum_taxonomy.json`)** — maps every question to an explicit `topic_id`, `subskill_id`, `educational_aim`, `exam_archetype`, and `difficulty` (`easy` / `medium` / `hard`).
4. **Non-Generic Figure Enforcement (`02c-figures.js` schemas)** — ensures that every diagram (`transformation_grid`, `nested_triangles`, `bowtie`, `right_triangle_altitude`, `shadow_similarity`, `angle_bisector`, `circle_chords_similarity`, `fn`, etc.) uses the **exact coordinates, side lengths, or functions** from the question stem and never leaks the answer on the diagram.
5. **T4 GPU Semantic Deduplication (`sentence-transformers`)** — computes cosine similarity across all stems in a domain so duplicate ideas are detected and replaced with fresh past-paper archetypes.
6. **Google Drive Checkpointing** — saves progress after every 10 questions so you never lose work if a session disconnects.

In [ ]:
# ── CELL 1: Install Dependencies & Mount Google Drive ──
!pip install -q -U sympy sentence-transformers pydantic

import os, re, json, time, shutil
import sympy as sp
from pathlib import Path
from google.colab import ai, drive

drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/ClipSAT_Bank_Pipeline')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('Available Colab AI models:', ai.list_models())

In [ ]:
# ── CELL 2: Clone or Pull Your ClipSAT Repository into Colab ──
REPO_DIR = Path('/content/ClipSAT')
if not REPO_DIR.exists():
    !git clone https://github.com/mohammedamy/ClipSAT.git /content/ClipSAT
else:
    !git -C /content/ClipSAT pull origin main

TAXONOMY_PATH = REPO_DIR / 'tools' / 'colab_bank_pipeline' / 'curriculum_taxonomy.json'
with open(TAXONOMY_PATH, 'r', encoding='utf-8') as f:
    TAXONOMY = json.load(f)

print('Loaded curriculum taxonomy tracks:', list(TAXONOMY.keys()))

In [ ]:
# ── CELL 3: ClipSAT Native Figure Schema Reference & Programmatic Verifier ──
# These match src/scripts/modules/02c-figures.js and scripts/validate-banks.js

FIGURE_SCHEMA_DOCS = '''
Supported ClipSAT `fig` JSON schemas (must match the exact numbers in the question stem, and MUST NOT print the final answer number on the diagram):
1. Coordinate Transformation Grid (for translations, reflections, rotations, dilations, symmetry):
   {"type": "transformation_grid", "xmin": -6, "xmax": 6, "ymin": -6, "ymax": 6, "object": [[1, 2], [4, 2], [2, 5]], "objectLabels": ["A", "B", "C"], "image": [[-4, 2], [-1, 2], [-3, 5]], "imageLabels": ["A'", "B'", "C'"], "mirrorLine": {"m": 1, "c": 0, "label": "y = x"}}
2. Nested Triangles / Thales Parallel Transversal:
   {"type": "nested_triangles", "labels": ["A", "B", "C", "D", "E"], "t": 0.45, "sides": {"ad": "6", "db": "4", "ae": "9", "ec": "x", "de": "12"}}
3. Bowtie / Hourglass Parallel Similar Triangles:
   {"type": "bowtie", "labels": ["A", "B", "C", "D", "E"], "ratio": 0.6, "sides": {"ab": "9", "cd": "15", "ae": "6", "ec": "x"}}
4. Right-Triangle Altitude to Hypotenuse (Geometric Mean Theorem):
   {"type": "right_triangle_altitude", "labels": ["A", "B", "C", "D"], "t": 0.36, "p": "4", "q": "9", "h": "h"}
5. Indirect Shadow / Mirror Similarity:
   {"type": "shadow_similarity", "obj1": {"label": "Post", "height": "6 ft", "shadow": "4 ft"}, "obj2": {"label": "Tower", "height": "h", "shadow": "28 ft"}}
6. Triangle Angle Bisector Theorem:
   {"type": "angle_bisector", "labels": ["A", "B", "C", "D"], "sides": {"ab": "10", "ac": "15", "bd": "6", "dc": "x"}}
7. Circle Intersecting Chords (Power of a Point):
   {"type": "circle_chords_similarity", "labels": ["A", "B", "C", "D", "P"], "sides": {"pa": "6", "pb": "8", "pc": "4", "pd": "x"}}
8. Side-by-Side Similar Polygons:
   {"type": "similar", "ratio": 0.65, "vertices": [[0,0],[8,0],[2,5]], "left": {"labels": ["A","B","C"], "sides": ["8","6","7"]}, "right": {"labels": ["D","E","F"], "sides": ["12","x","10.5"]}}
9. Function / Coordinate Graph (`fn`):
   {"type": "fn", "fns": ["x*x - 4*x + 3", "x - 1"], "xlim": [-2, 6], "ylim": [-3, 6], "points": [{"x": 1, "y": 0, "label": "(1, 0)"}]}
10. Standard Triangle / Circle:
   {"type": "triangle", "vertices": [[30, 160], [210, 160], [80, 40]], "labels": ["A", "B", "C"], "sides": ["14", "10", "12"], "angles": ["52°", "", ""]}
   {"type": "circle", "r": 6, "chord": true, "sector": 60}
'''

def extract_json_object(raw_text: str) -> dict:
    """Extracts a JSON object from model markdown or raw output."""
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw_text.strip(), flags=re.MULTILINE)
    match = re.search(r'\{[\s\S]*\}', cleaned)
    if not match:
        raise ValueError('No JSON object found in model output.')
    return json.loads(match.group(0))

def verify_item_programmatically(item: dict) -> tuple[bool, str]:
    """Verifies choices, SymPy proof, and non-generic figure alignment."""
    stem = str(item.get('text') or item.get('q') or '').strip()
    choices = item.get('choices', [])
    correct = item.get('correct')
    solution = str(item.get('solution', '')).strip()
    sympy_code = str(item.get('sympy_check', '')).strip()

    if len(stem) < 25:
        return False, 'Stem is too short.'
    if not isinstance(choices, list) or len(choices) != 4 or len(set(choices)) != 4:
        return False, 'Must have exactly 4 distinct choices.'
    if not isinstance(correct, int) or not (0 <= correct <= 3):
        return False, 'correct must be an integer 0..3.'
    if len(solution) < 40:
        return False, 'Solution must be at least 40 characters with full steps.'

    # Execute SymPy verification code generated by the model
    if sympy_code:
        ns = {'sp': sp}
        try:
            exec(sympy_code, {'__builtins__': {'abs': abs, 'round': round, 'min': min, 'max': max, 'sum': sum, 'len': len, 'range': range, 'int': int, 'float': float}}, ns)
        except Exception as e:
            return False, f'SymPy verification failed: {e}'
    else:
        return False, 'Missing sympy_check verification snippet.'

    # Check figure validity & ensure it is NOT generic and does NOT leak the answer
    fig = item.get('fig')
    if fig and isinstance(fig, dict):
        fig_json_str = json.dumps(fig)
        stem_nums = set(re.findall(r'-?\d+(?:\.\d+)?', stem))
        fig_nums = set(re.findall(r'-?\d+(?:\.\d+)?', fig_json_str))
        if stem_nums and fig_nums and not (stem_nums & fig_nums):
            return False, f'Generic figure rejected: figure numbers {fig_nums} do not match stem numbers {stem_nums}.'

        # Ensure pure trig fn graphs mention graph/curve/period/amplitude (validate-banks.js rule)
        if fig.get('type') in ('fn', 'function_graph') and ('\\sin' in stem or '\\cos' in stem):
            if not re.search(r'graph|curve|period|amplitude|sketch|figure|shown', stem, re.I):
                return False, 'Trig function graph stem must reference the graph/curve/period/amplitude.'

    return True, 'OK'

print('Verifier ready.')

In [ ]:
# ── CELL 4: GPU Semantic Deduplication (Sentence-Transformers) ──
import torch
from sentence_transformers import SentenceTransformer, util

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f'SentenceTransformer loaded on {device}')

def find_duplicate_indices(pool: list[dict], similarity_threshold: float = 0.86) -> set[int]:
    """Finds indices of questions whose stem is semantically redundant with an earlier question."""
    stems = [str(q.get('text') or q.get('q') or '') for q in pool]
    if not stems:
        return set()
    embs = embedder.encode(stems, convert_to_tensor=True, show_progress_bar=True)
    sim_matrix = util.cos_sim(embs, embs)
    dups = set()
    for i in range(len(stems)):
        if i in dups:
            continue
        for j in range(i + 1, len(stems)):
            if float(sim_matrix[i][j]) >= similarity_threshold:
                dups.add(j)
    return dups

In [ ]:
# ── CELL 5: Colab AI Auditor & Past-Paper Question Generator ──
# Uses google.colab.ai with your Ultra subscription.

MODEL_NAME = 'google/gemini-3.8-flash'  # Or 'google/gemini-3.1-pro-preview' for deepest reasoning

def audit_or_generate_item(
    existing_q: dict,
    track_id: str,
    domain_spec: dict,
    subskill: dict,
    force_rewrite: bool = False,
    avoid_stems: list[str] = None,
) -> dict:
    avoid_block = ''
    if avoid_stems:
        sample_avoid = '\n'.join(f'- {s[:120]}' for s in avoid_stems[-8:])
        avoid_block = f'\nDO NOT repeat the scenario or mathematical setup of these recent items:\n{sample_avoid}\n'

    prompt = f'''You are a Senior Mathematics Exam Author and Psychometric Auditor for {track_id.upper()}.

TARGET CURRICULUM SPECIFICATION:
- Domain: {domain_spec["domain"]}
- Topic ID: {domain_spec["topic_id"]}
- Subskill ID: {subskill["id"]}
- Educational Aim: {subskill["aim"]}
- Past-Paper Archetype: {subskill["archetype"]}
- Target Difficulty: {subskill["difficulty"]}
- Allowed Figure Types: {json.dumps(domain_spec["allowed_figures"])}
{avoid_block}
{FIGURE_SCHEMA_DOCS}

EXISTING QUESTION:
{json.dumps(existing_q, ensure_ascii=False)}

TASK:
{"Rewrite this question from scratch to strictly test the Target Subskill and Past-Paper Archetype." if force_rewrite else "Audit the existing question. If it is mathematically flawed, off-topic for the Educational Aim, uses a generic figure, or has weak distractors, rewrite it completely."}

Return ONLY a single valid JSON object with these exact keys:
{{
  "action": "kept" or "rewritten",
  "id": "{existing_q.get('id', '')}",
  "domain": "{domain_spec['domain']}",
  "topic_id": "{domain_spec['topic_id']}",
  "subskill_id": "{subskill['id']}",
  "educational_aim": "{subskill['aim']}",
  "exam_archetype": "{subskill['archetype']}",
  "difficulty": "{subskill['difficulty']}",
  "text": "Question stem using \\( ... \\) for inline LaTeX.",
  "choices": ["Choice A", "Choice B", "Choice C", "Choice D"],
  "correct": 0,
  "solution": "Step-by-step mathematical derivation showing why the correct choice holds and why common misconceptions fail.",
  "sympy_check": "Python code using sympy (already imported as sp) that computes the exact answer and ends with `assert ...`",
  "fig": null or one of the Allowed Figure Types JSON objects matching the exact numbers in `text` (WITHOUT revealing the final answer on the diagram)
}}
'''

    last_err = ''
    for attempt in range(4):
        raw = ai.generate_text(prompt if not last_err else prompt + f'\n\nPREVIOUS ATTEMPT FAILED VERIFICATION: {last_err}\nFix the JSON and SymPy check.', model_name=MODEL_NAME)
        try:
            candidate = extract_json_object(raw)
            ok, reason = verify_item_programmatically(candidate)
            if ok:
                # Remove temporary sympy_check before saving to production bank if desired, or keep in audit log
                return candidate
            last_err = reason
        except Exception as e:
            last_err = str(e)
        time.sleep(0.5)

    raise RuntimeError(f"Could not verify item {existing_q.get('id')} after 4 attempts: {last_err}")

In [ ]:
# ── CELL 6: Run the Pipeline on a Track with Google Drive Checkpointing ──
TRACK_TO_RUN = 'geo'  # Change to 'sat', 'act', 'calculus', etc.
MAX_ITEMS_TO_PROCESS = 30  # Set to None to process the entire bank, or e.g. 30 to test one topic batch first

bank_path = REPO_DIR / 'bank-data' / f'{TRACK_TO_RUN}.json'
checkpoint_path = WORK_DIR / f'{TRACK_TO_RUN}_verified_checkpoint.json'

with open(bank_path, 'r', encoding='utf-8') as f:
    bank_data = json.load(f)

pool = bank_data.get('pool', [])
print(f'Loaded {len(pool)} items from {bank_path.name}')

# Find semantic duplicates on GPU
dup_indices = find_duplicate_indices(pool, similarity_threshold=0.86)
print(f'Detected {len(dup_indices)} semantically redundant items to rewrite with fresh past-paper ideas.')

track_domains = TAXONOMY[TRACK_TO_RUN]['domains']
recent_stems = []

limit = len(pool) if MAX_ITEMS_TO_PROCESS is None else min(len(pool), MAX_ITEMS_TO_PROCESS)
updated_count = 0

for idx in range(limit):
    q = pool[idx]
    # Map question to domain & subskill round-robin or by existing domain
    dom_spec = track_domains[idx % len(track_domains)]
    subskills = dom_spec['subskills']
    subskill = subskills[(idx // len(track_domains)) % len(subskills)]

    force_rewrite = (idx in dup_indices) or ('topic_id' not in q)
    try:
        verified_q = audit_or_generate_item(
            existing_q=q,
            track_id=TRACK_TO_RUN,
            domain_spec=dom_spec,
            subskill=subskill,
            force_rewrite=force_rewrite,
            avoid_stems=recent_stems,
        )
        # Keep clean production fields + taxonomy metadata for future updates
        clean_q = {
            'id': verified_q.get('id') or q.get('id') or f'{TRACK_TO_RUN}-{idx+1:04d}',
            'domain': verified_q['domain'],
            'topic_id': verified_q['topic_id'],
            'subskill_id': verified_q['subskill_id'],
            'educational_aim': verified_q['educational_aim'],
            'exam_archetype': verified_q['exam_archetype'],
            'difficulty': verified_q['difficulty'],
            'text': verified_q['text'],
            'choices': verified_q['choices'],
            'correct': verified_q['correct'],
            'solution': verified_q['solution'],
        }
        if verified_q.get('fig'):
            clean_q['fig'] = verified_q['fig']

        pool[idx] = clean_q
        recent_stems.append(clean_q['text'])
        updated_count += 1
        print(f'[{idx+1}/{limit}] ✅ Verified {clean_q["id"]} | {clean_q["subskill_id"]} ({clean_q["difficulty"]}) | fig={clean_q.get("fig", {}).get("type", "none")}')
    except Exception as err:
        print(f'[{idx+1}/{limit}] ⚠️ Skipped {q.get("id")}: {err}')

    # Checkpoint to Google Drive every 10 items
    if (idx + 1) % 10 == 0 or (idx + 1) == limit:
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(bank_data, f, ensure_ascii=False, indent=2)
        print(f'  💾 Saved checkpoint to {checkpoint_path}')

# Copy verified checkpoint back to repo and run ClipSAT's official validator
shutil.copy(checkpoint_path, bank_path)
!node /content/ClipSAT/scripts/validate-banks.js